In [10]:
from google.colab import files
uploaded = files.upload()

Saving 2023.xlsx to 2023 (2).xlsx
Saving 2024.xlsx to 2024 (2).xlsx
Saving 2025.xlsx to 2025 (2).xlsx


In [11]:
!pip install openpyxl

In [12]:
import pandas as pd
import numpy as np

file_2023 = "2023.xlsx"
file_2024 = "2024.xlsx"
file_2025 = "2025.xlsx"

In [13]:
def extract_sundry_debtors(file):
    df = pd.read_excel(file, header=None)

    # Find row containing "Sundry Debtors"
    sundry_index = df[df.apply(lambda r: r.astype(str)
                               .str.contains("Sundry Debtors", case=False, na=False)
                               .any(), axis=1)].index

    if len(sundry_index) == 0:
        print(f"Sundry Debtors not found in {file}")
        return pd.DataFrame(columns=["Name", "Amount"])

    start_row = sundry_index[0] + 1
    extracted = []

    for i in range(start_row, len(df)):
        row = df.iloc[i]

        # Stop if Total line found
        if row.astype(str).str.contains("Total", case=False, na=False).any():
            break

        name = None
        amount = None

        # Get first text cell as name
        for cell in row:
            if isinstance(cell, str) and cell.strip() != "":
                name = cell.strip()
                break

        # Get last numeric value in row as amount
        numeric_values = [x for x in row if isinstance(x, (int, float))]
        if len(numeric_values) > 0:
            amount = float(numeric_values[-1])

        if name and amount is not None:
            extracted.append([name, amount])

    return pd.DataFrame(extracted, columns=["Name", "Amount"])

In [14]:
data_2023 = extract_sundry_debtors(file_2023)
data_2024 = extract_sundry_debtors(file_2024)
data_2025 = extract_sundry_debtors(file_2025)

data_2023.rename(columns={"Amount": "2023"}, inplace=True)
data_2024.rename(columns={"Amount": "2024"}, inplace=True)
data_2025.rename(columns={"Amount": "2025"}, inplace=True)

In [15]:
merged = pd.merge(data_2023, data_2024, on="Name", how="outer")
merged = pd.merge(merged, data_2025, on="Name", how="outer")

merged.fillna(0, inplace=True)

In [16]:
def classify(row):
    y1, y2, y3 = row["2023"], row["2024"], row["2025"]

    if y1 > 0 and y2 == 0 and y3 == 0:
        return "Fully Paid"

    if y3 < y2 or y2 < y1:
        return "Partially Paid"

    if y1 == y2 == y3:
        return "Not Paid"

    if y3 > y2 or y2 > y1:
        return "Increased (New Credit Given)"

    return "Check Manually"

merged["Status"] = merged.apply(classify, axis=1)

# Detect new persons
merged["New in 2024"] = np.where((merged["2023"] == 0) & (merged["2024"] > 0), "Yes", "No")
merged["New in 2025"] = np.where((merged["2024"] == 0) & (merged["2025"] > 0), "Yes", "No")

merged = merged.sort_values("Name")

In [17]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment

output_file = "Final_Sundry_Debtors_Analysis.xlsx"

merged.columns = [
    "Debtor Name",
    "Amount 2023",
    "Amount 2024",
    "Amount 2025",
    "Payment Status",
    "New in 2024",
    "New in 2025"
]

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    merged.to_excel(writer, index=False, sheet_name='Analysis', startrow=2)

wb = load_workbook(output_file)
ws = wb["Analysis"]

ws["A1"] = "Sundry Debtors Analysis (2023–2025)"
ws["A1"].font = Font(size=14, bold=True)
ws.merge_cells("A1:G1")
ws["A1"].alignment = Alignment(horizontal="center")

for cell in ws[3]:
    cell.font = Font(bold=True)

wb.save(output_file)

print("Final Excel file created successfully!")

Final Excel file created successfully!


In [18]:
from google.colab import files
files.download("Final_Sundry_Debtors_Analysis.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>